<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/sentiment/Notebook_1_Manual_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 · How do customers feel about their box?
**NTU MSc Growth Hacking | Customer Insight Lab**

The Oddbox founder wants to know what might encourage households to order again. A colleague suggests counting happy customers. What might that count hide?

You will prepare a small set of fictional practice reviews, judge the feeling expressed in each one, and use a customer's words to propose a test. The exercise includes all the data you need.

**Working in Colab** — Save your own copy first. Work down the page using the play button beside each code cell. Double-click a text cell to write in it. Keep your individual answers before discussing them with a partner.

## 1 · What are you judging?
**Sentiment** describes the attitude expressed in the words. A **topic** tells you what the customer is talking about. Two people can both mention delivery and express very different feelings.

Read the whole review and choose the label that best describes the customer’s overall feeling.

| Label | When to choose it |
|---|---|
| Positive | The customer sounds pleased overall. |
| Negative | The customer sounds disappointed overall. |
| Mixed (neutral) | Praise and criticism are balanced, or the customer gives no clear opinion. |

For example, “The recipe card helped me use every leek. Please send another!” sounds **positive**. The phrase “Please send another!” shows approval.

## 2 · Which rows should count?
Each source ID represents one posted review. Inspect the import below. Find a row with no words and a review imported twice. What would happen to the share of positive reviews if a positive review appeared twice? Would identical wording from two different people be enough evidence to delete one?

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', None)
raw = pd.DataFrame([["P01","I used every carrot this week. Delighted with the box!"],["P02","Both pears were bruised. I asked for a refund."],["P03","My box arrived on Thursday and contained six items."],["P04","Lovely veg, but the second cabbage went straight into the bin."],["P05","Great. Another box left in the rain 🙄"],["P06","I wasn’t expecting much, but I’ve already ordered again."],["P07","The veg was fresh, but missing two deliveries is enough for me to cancel."],["P08","Can I see next Thursday’s contents before I order?"],["P09","Not bad at all! The kids even ate the beetroot."],["P10","I liked the surprise as much as I disliked having to change my meal plan."],["P01","  I used every carrot this week. Delighted with the box!  "],["P11","    "]], columns=['source_id', 'original'])
display(raw)

This cell makes a reading copy and trims spaces at the edges. A blank row has no opinion to classify. Counting it as neutral would inflate that group. We remove repeated copies only when the source ID and trimmed words both match. Keeping the original lets us check every decision later. Predict the number of usable reviews before running the cell.

In [ ]:
prepared = raw.assign(review=raw.original.fillna('').str.strip())
blank = prepared.review.eq('')
repeated = prepared.duplicated(['source_id', 'review']) & ~blank
display(prepared[blank | repeated].assign(reason=lambda d: d.review.map(
    lambda words: 'Repeated import' if words else 'Blank text')))
reviews = prepared[~(blank | repeated)].set_index('source_id')
print(f'{len(raw)} imported → {len(reviews)} usable reviews')

### Which words should we keep?
Read “Not bad at all!” aloud, then imagine a cleaning rule that removes “not”. What judgement might someone now make? In P05, cover the emoji and reread the first word. For this activity, keep punctuation, capital letters, emojis and small words in the reading copy.

Later, when grouping reviews by **topic**, we may want related words to count together.

**Stemming** cuts words back to a common stem, so “delivering” and “delivered” can both become “deliv”.

**Lemmatisation** uses vocabulary and grammar to find a dictionary form, so the same two verbs can become “deliver”.

Both can help us find repeated subjects. Neither tells us whether the delivery went well.

Which exact words in P07 would you want to preserve before simplifying its language?

## 3 · How does each customer feel?
Run the cell below to see the ten reviews. Read each one and choose **one label** from the menu beside it. If a review mentions something good and something bad, ask which feeling comes through most strongly. Choose **Mixed (neutral)** if they feel balanced or the writer simply reports a fact.

When you have labelled all ten, move on to the pair discussion.

In [ ]:
import ipywidgets as widgets

labels = ['Positive', 'Negative', 'Mixed (neutral)']
choices = {}
for review_id, words in reviews.review.items():
    print(f'{review_id} · {words}')
    menu = widgets.Dropdown(options=['Choose a label'] + labels)
    choices[review_id] = menu
    display(menu)

## 4 · Compare decisions in pairs
Look at the labels you each chose for the same ten reviews.

Find one where you chose different labels. Read the review aloud and ask each other:

“Which words led you to choose that label?”

If you agreed on all ten, discuss the review you found hardest to judge.

In [ ]:
# Example entry format inside the braces: 'P04': 'Mixed (neutral)'
changes = {}
sheet = reviews[['review']].copy()
sheet['individual'] = [choices[key].value for key in sheet.index]
if not sheet.individual.isin(labels).all():
    print('Choose a label for every review in Block 3, then run this cell again.')
elif any(key not in sheet.index or value not in labels for key, value in changes.items()):
    print('Check each changed ID and label.')
else:
    results = sheet.copy()
    results['agreed'] = [changes.get(key, label) for key, label in results.individual.items()]
    display(results)
    counts = results[['individual', 'agreed']].apply(
        lambda column: column.value_counts().reindex(labels, fill_value=0))
    display(counts)
    results.to_csv('manual_sentiment.csv', index_label='source_id')
    print('Your completed sheet is available as manual_sentiment.csv in the Colab Files panel.')

## 5 · What would you tell the founder?
Use your agreed counts and one quotation to write a three-sentence briefing. Give the count **out of ten**, identify one possible reason to order again or cancel, and propose one change to test. Name the behaviour you would measure. A review can suggest why someone feels a certain way. To establish whether a change affects repeat orders, you need to observe what customers actually do.

**Our briefing:** …

**Challenge your conclusion.** If one mixed review moved to a different label, how many percentage points would the positive share change if it entered or left the positive group? Why would it be risky to generalise from these fictional examples to Oddbox customers?

**Carry into Notebook 2.** Predict which of sarcasm, negation or mixed feelings will be hardest for an automated method. Choose a review ID and explain what the method would need to pick up.

<details><summary>Open after discussion · Tutor debrief</summary>

P03 reports facts. P08 asks for information. Both can be coded Mixed (neutral), although P08 still suggests a useful research question about seeing box contents. P05 uses “Great” sarcastically. The rain and eye-roll support a negative reading. P07 contains praise and criticism, but the decision to cancel supports a negative overall label. P10 explicitly balances the two sides, so the suggested rule gives Mixed (neutral). P04 is a useful disagreement to explore because the writer describes waste without giving an overall verdict.

One review represents ten percentage points in this practice set. Two pairs could produce the same totals while disagreeing on individual reviews. Ask them to show which words informed their decisions.

</details>